# Fin API playground

A guided tour of two live APIs from one portfolio project, showing what they do under concurrency:

- **[fin-order-management-api](https://github.com/SaaSGun/fin-order-management-api)**: the transactional side. Orders, pre-trade compliance, fills and settlement on PostgreSQL.
- **[fin-analytics-api](https://github.com/SaaSGun/fin-analytics-api)**: the analytics side. Holdings, sector exposures, performance and ESG from BigQuery.

Every cell calls the deployed services. Run **Setup** first, then the sections in order. The order-management scenarios run on an account of your own, so nothing you do here affects anyone else.

## Setup

Your invitation email has three values: an OMS API key, your OMS client id, and an analytics API key. Add them as Colab secrets (🔑 in the left sidebar) named `OMS_API_KEY`, `OMS_CLIENT_ID` and `ANALYTICS_API_KEY`, or paste each one when prompted. The keys expire on their own after a week.

In [ ]:
%pip install -q httpx

In [ ]:
import asyncio
import os
import random
import time
import uuid
from collections import Counter
from getpass import getpass

import httpx

OMS_URL = os.environ.get("OMS_URL", "https://fin-oms-api-miatwtkcba-ez.a.run.app")
ANALYTICS_URL = os.environ.get("ANALYTICS_URL", "https://fin-analytics-api-1089997989086.europe-west4.run.app")


def setting(name):
    """A Colab secret if there is one, else an environment variable, else a prompt."""
    try:
        from google.colab import userdata

        return userdata.get(name)
    except Exception:
        return os.environ.get(name) or getpass(f"{name}: ")


OMS_API_KEY = setting("OMS_API_KEY")
OMS_CLIENT_ID = setting("OMS_CLIENT_ID")
ANALYTICS_API_KEY = setting("ANALYTICS_API_KEY")

oms = httpx.AsyncClient(
    base_url=OMS_URL, headers={"X-API-Key": OMS_API_KEY}, timeout=60, limits=httpx.Limits(keepalive_expiry=60)
)
analytics = httpx.AsyncClient(base_url=ANALYTICS_URL, headers={"X-API-Key": ANALYTICS_API_KEY}, timeout=60)


async def call(client, method, url, **kwargs):
    """One request. On 429 or 503 it waits as long as Retry-After says, then retries (3 times at most)."""
    for attempt in range(4):
        response = await client.request(method, url, **kwargs)
        if response.status_code not in (429, 503) or attempt == 3:
            return response
        await asyncio.sleep(float(response.headers.get("Retry-After", "1")))


async def ok(client, method, url, **kwargs):
    """call(), but raise unless it succeeded; returns the response's `data`."""
    response = await call(client, method, url, **kwargs)
    if response.is_error:
        raise RuntimeError(f"{method} {url} -> {response.status_code}: {response.text}")
    return response.json()["data"]


def new_key():
    return {"Idempotency-Key": str(uuid.uuid4())}


def order(instrument_id, quantity, price, side="BUY"):
    return {
        "account_id": ACCOUNT_ID,
        "instrument_id": instrument_id,
        "side": side,
        "order_type": "MARKET",
        "quantity": str(quantity),
        "reference_price": str(price),
    }


async def wait_for_rate_limit():
    # Each key may send 5 requests/s, in bursts of up to 20. Before firing a
    # burst of simultaneous requests, let that allowance refill, so the race
    # isn't cut short by our own rate limit.
    await asyncio.sleep(4)


async def cancel(order_id):
    etag = (await call(oms, "GET", f"/orders/{order_id}")).headers["ETag"]
    return await ok(oms, "POST", f"/orders/{order_id}/cancel", headers={**new_key(), "If-Match": etag})


accounts = await ok(oms, "GET", f"/clients/{OMS_CLIENT_ID}/accounts")
ACCOUNT_ID = next(a["id"] for a in accounts if a["status"] == "ACTIVE")
instruments = {i["symbol"]: i["id"] for i in await ok(oms, "GET", "/instruments")}
AAPL, ASML, RESTRICTED = instruments["AAPL"], instruments["ASML"], instruments["RESTRICTEDCO"]

print("your account:", ACCOUNT_ID)
for b in await ok(oms, "GET", f"/accounts/{ACCOUNT_ID}/cash-balances"):
    print(f"  {b['currency']}: balance {b['balance']}, reserved {b['reserved_amount']}")

## Part 1: order management

### 1. Place an order

Placing a BUY reserves its cash straight away (quantity × reference price). Two orders therefore can't spend the same euros, even when they arrive at the same moment. Each pre-trade compliance rule's result is stored with the order.

In [ ]:
placed = await ok(oms, "POST", "/orders", json=order(ASML, 5, "500.00"), headers=new_key())
print(placed["status"], "- reserved", placed["reserved_amount"], placed["currency"])
for check in await ok(oms, "GET", f"/orders/{placed['id']}/compliance-checks"):
    print(f"  {check['rule_type']:<22} {check['result']}  {check['detail'] or ''}")

### 2. A compliance rejection is still a `201`

`RESTRICTEDCO` is on the restricted list. The order is accepted and evaluated, and fails a rule. The response says so in its body, with the reason, not with an error status.

In [ ]:
rejected = await ok(oms, "POST", "/orders", json=order(RESTRICTED, 10, "50.00"), headers=new_key())
print(rejected["status"], "-", rejected["rejection_reason"])

### 3. Retries are safe: idempotency keys

Every write needs an `Idempotency-Key`:
- the same key with the same body returns the stored response;
- the same key with a different body gets `409`;
- a missing key gets `422`.

Even ten *simultaneous* copies of one request create a single order. The first takes a Postgres advisory lock on the key, and the rest wait, then replay its response.

In [ ]:
key = new_key()
first = await call(oms, "POST", "/orders", json=order(ASML, 1, "500.00"), headers=key)
retry = await call(oms, "POST", "/orders", json=order(ASML, 1, "500.00"), headers=key)
changed = await call(oms, "POST", "/orders", json=order(ASML, 2, "500.00"), headers=key)
missing = await call(oms, "POST", "/orders", json=order(ASML, 1, "500.00"))
print("first         ", first.status_code, first.json()["data"]["id"])
print("retry         ", retry.status_code, retry.json()["data"]["id"], "<- the same order")
print("changed body  ", changed.status_code, changed.json()["detail"])
print("no key        ", missing.status_code, missing.json()["detail"])

await wait_for_rate_limit()
key = new_key()
copies = await asyncio.gather(*[oms.post("/orders", json=order(ASML, 1, "500.00"), headers=key) for _ in range(10)])
orders = {r.json()["data"]["id"] for r in copies if r.status_code == 201}
print("10 simultaneous copies:", [r.status_code for r in copies], "->", len(orders), "order")

### 4. Ten portfolio managers race for the last of a limit

Your account may hold at most 50 AAPL. The concentration limit counts shares held *and* open BUY orders. Ten 20-share orders arrive at once: each fits on its own, but only two fit together. Every BUY locks the limit row, so they're evaluated one at a time, and exactly two are admitted.

Before this was fixed, all ten could be admitted. The cell cancels the admitted orders at the end, so you can run it again.

In [ ]:
await wait_for_rate_limit()
racers = await asyncio.gather(
    *[oms.post("/orders", json=order(AAPL, 20, "150.00"), headers=new_key()) for _ in range(10)]
)
results = [r.json()["data"] for r in racers]
print(Counter(o["status"] for o in results))
print("why the others were rejected:", next(o["rejection_reason"] for o in results if o["status"] == "REJECTED"))

for o in results:
    if o["status"] != "REJECTED":
        await cancel(o["id"])  # give the limit back

### 5. Two people cancel the same order: optimistic concurrency

`GET /orders/{id}` returns an `ETag`, which is the order's version. A cancel must send it back in `If-Match`. Fire ten cancels with the same ETag: one wins. The other nine get `412 Precondition Failed`, because the version they saw no longer exists. A cancel without `If-Match` gets `428`.

In [ ]:
target = await ok(oms, "POST", "/orders", json=order(ASML, 1, "500.00"), headers=new_key())
etag = (await call(oms, "GET", f"/orders/{target['id']}")).headers["ETag"]

await wait_for_rate_limit()
cancels = await asyncio.gather(
    *[oms.post(f"/orders/{target['id']}/cancel", headers={**new_key(), "If-Match": etag}) for _ in range(10)]
)
print("10 simultaneous cancels:", sorted(r.status_code for r in cancels))

no_etag = await call(oms, "POST", f"/orders/{target['id']}/cancel", headers=new_key())
print("without If-Match:", no_etag.status_code, no_etag.json()["detail"])

### 6. A fill is one transaction

Recording an execution does all of this in a single commit:
- updates the order;
- releases its cash reservation;
- books the position at average cost;
- moves the cash;
- creates the settlement.

Executions need the `admin` scope. Your key has it, so you can play the market's part.

In [ ]:
async def eur_and_asml():
    eur = next(b for b in await ok(oms, "GET", f"/accounts/{ACCOUNT_ID}/cash-balances") if b["currency"] == "EUR")
    held = next((p["quantity"] for p in await ok(oms, "GET", f"/accounts/{ACCOUNT_ID}/positions")
                 if p["instrument_id"] == ASML), "0")
    return f"EUR balance {eur['balance']}, reserved {eur['reserved_amount']} | ASML held {held}"


buy = await ok(oms, "POST", "/orders", json=order(ASML, 10, "500.00"), headers=new_key())
print("before the fill:", await eur_and_asml())
execution = await ok(
    oms, "POST", f"/orders/{buy['id']}/executions", json={"quantity": "10", "price": "498.50"}, headers=new_key()
)
print("after the fill: ", await eur_and_asml())
history = await ok(oms, "GET", f"/orders/{buy['id']}/status-history")
print("order status history:", " -> ".join(h["status"] for h in history))

### 7. Settle it

The fill created a settlement, which is versioned like the order: settling it needs its `ETag` in `If-Match`.

In [ ]:
found = await call(oms, "GET", f"/executions/{execution['id']}/settlement")
settlement = found.json()["data"]
print("settlement:", settlement["status"], "- expected on", settlement["expected_settlement_date"])

settled = await ok(
    oms, "POST", f"/settlements/{settlement['id']}/settle", json={},
    headers={**new_key(), "If-Match": found.headers["ETag"]},
)
print("settlement:", settled["status"], "at", settled["settled_at"])

### 8. Rate limiting

Twenty-five requests at once from one key. Roughly the first 20 (the burst allowance) succeed, and the rest get `429` with a `Retry-After` header.

In [ ]:
await wait_for_rate_limit()
burst = await asyncio.gather(*[oms.get("/instruments") for _ in range(25)])
print(Counter(r.status_code for r in burst))
limited = next((r for r in burst if r.status_code == 429), None)
if limited is not None:
    print("Retry-After:", limited.headers["Retry-After"], "s")

### 9. Clean up

Cancels whatever is still open on your account, which releases its reserved cash.

In [ ]:
still_open = [
    o for o in await ok(oms, "GET", "/orders", params={"account_id": ACCOUNT_ID, "limit": 100})
    if o["status"] in ("NEW", "PARTIALLY_FILLED")
]
for o in still_open:
    await cancel(o["id"])
print(f"cancelled {len(still_open)} open orders")
for b in await ok(oms, "GET", f"/accounts/{ACCOUNT_ID}/cash-balances"):
    print(f"  {b['currency']}: balance {b['balance']}, reserved {b['reserved_amount']}")

## Part 2: portfolio analytics

The read side of the same domain, served from a BigQuery warehouse. Reference data needs the `standard:read` scope, and anything about portfolio positions needs `confidential:read`. Your key has both.

### 10. The AAPL you just traded, from the warehouse

In [ ]:
security = await ok(analytics, "GET", "/securities/AAPL")
for field in ("company_name", "gics_sector", "market_cap", "trailing_pe", "beta", "last_close", "last_trade_date",
              "total_esg_risk_score", "esg_risk_category"):
    print(f"  {field:<22} {security[field]}")

print("last 5 closes:")
for p in await ok(analytics, "GET", "/securities/AAPL/prices", params={"limit": 5}):
    print(f"  {p['trade_date']}  {p['close']}")

### 11. A portfolio: holdings, sector tilts, performance

In [ ]:
portfolios = await ok(analytics, "GET", "/portfolios")
for p in portfolios:
    print(f"  {p['portfolio_id']:<12} {p['strategy']:<28} AUM ${p['aum_usd']:,.0f}  benchmark {p['benchmark_id']}")

portfolio = portfolios[0]["portfolio_id"]
print(f"\n{portfolio} - top holdings:")
for h in await ok(analytics, "GET", f"/portfolios/{portfolio}/holdings", params={"limit": 5}):
    print(f"  {h['ticker']:<6} {h['weight']:6.1%}  {h['gics_sector']}")

print(f"\n{portfolio} - sector tilts vs benchmark:")
for e in await ok(analytics, "GET", f"/portfolios/{portfolio}/exposures", params={"limit": 5}):
    print(f"  {e['gics_sector']:<26} {e['active_weight']:+6.1%}")

# Performance comes oldest first, so follow the cursor to the latest day.
params = {"limit": 500}
while True:
    page = (await call(analytics, "GET", f"/portfolios/{portfolio}/performance", params=params)).json()
    cursor = page["meta"]["pagination"]["next_cursor"]
    if not cursor:
        break
    params["cursor"] = cursor
last = page["data"][-1]
print(f"\ncumulative return to {last['trade_date']}: portfolio {last['portfolio_cum_return']:+.1%}, "
      f"benchmark {last['benchmark_cum_return']:+.1%}")

### 12. Cursor pagination

Lists are paged with an opaque cursor instead of `OFFSET`. Following `next_cursor` never skips or repeats a row, even while data changes.

In [ ]:
page_1 = (await call(analytics, "GET", f"/portfolios/{portfolio}/exposures", params={"limit": 3})).json()
cursor = page_1["meta"]["pagination"]["next_cursor"]
page_2 = (await call(analytics, "GET", f"/portfolios/{portfolio}/exposures", params={"limit": 3, "cursor": cursor})).json()
print("page 1:", [e["gics_sector"] for e in page_1["data"]])
print("page 2:", [e["gics_sector"] for e in page_2["data"]])
print("meta:  ", page_2["meta"]["pagination"])

### 13. ESG: the portfolio's risk, and what drives it

In [ ]:
summary = await ok(analytics, "GET", f"/portfolios/{portfolio}/esg-summary")
print(f"weighted ESG risk: portfolio {summary['portfolio_weighted_esg_risk_score']:.1f}, "
      f"benchmark {summary['benchmark_weighted_esg_risk_score']:.1f}")

print("biggest contributors to ESG risk:")
for laggard in await ok(analytics, "GET", f"/portfolios/{portfolio}/esg-laggards", params={"limit": 5}):
    print(f"  {laggard['ticker']:<6} score {laggard['total_esg_risk_score']:5.1f}  "
          f"contribution {laggard['weighted_risk_contribution']:.2f}")

### 14. Query results are cached

Every cache miss is a billed BigQuery query, so results are cached in the service. The same screen runs twice below. The first call goes to BigQuery; the second comes from the cache. A random threshold makes sure the first call is a genuine miss.

In [ ]:
params = {"sector": "Information Technology", "max_risk_score": round(random.uniform(20, 30), 2), "limit": 5}
for attempt in ("first call ", "second call"):
    start = time.perf_counter()
    screen = await ok(analytics, "GET", "/esg/screen", params=params)
    print(f"{attempt}: {(time.perf_counter() - start) * 1000:5.0f} ms, {len(screen)} results")
for s in screen:
    print(f"  {s['ticker']:<6} ESG risk {s['total_esg_risk_score']}  {s['company_name']}")

### 15. Authentication errors

In [ ]:
for name, client, path in (("OMS", oms, "/instruments"), ("analytics", analytics, "/portfolios")):
    response = await client.get(path, headers={"X-API-Key": "not-a-real-key"})
    print(f"{name:<10} {response.status_code} {response.json()['detail']}")

## Where to look next

- The load tests behind these behaviours, run against the same deployments: [OMS](https://github.com/SaaSGun/fin-order-management-api#load-testing--and-what-it-found) and [analytics](https://github.com/SaaSGun/fin-analytics-api#performance).
- Every endpoint, with its schema and required scope: [OMS `/docs`](https://fin-oms-api-miatwtkcba-ez.a.run.app/docs) and [analytics `/docs`](https://fin-analytics-api-1089997989086.europe-west4.run.app/docs).